# Early Exit Adapter Pipeline

Colab-friendly notebook wrapper around the `early_exit_adapters` package.

## 1. Clone repo and enter the Colab working folder

Set `REPO_URL` to the GitHub repo that contains this package. The cell clones into `/content/Speculative_decoding`, updates it if it already exists, changes the process working directory, and verifies the package is present.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/YOUR_GITHUB_USER/Speculative_decoding.git"  # <-- change this
REPO_DIR = Path("/content/Speculative_decoding")

if "YOUR_GITHUB_USER" in REPO_URL:
    raise ValueError("Set REPO_URL to your actual GitHub repo URL before running this cell.")

if REPO_DIR.exists():
    print(f"Repo already exists at {REPO_DIR}; pulling latest changes...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("cwd:", Path.cwd())
print("top-level files:", sorted(p.name for p in Path.cwd().iterdir())[:20])

assert Path("early_exit_adapters").is_dir(), "Missing early_exit_adapters package. Are you in the cloned repo root?"
assert Path("notebooks/run_full_pipeline.ipynb").exists(), "Notebook file not found in expected location."
print("Colab folder check passed.")

## 2. Install/import dependencies

In [ ]:
%pip install -U torch transformers datasets pandas matplotlib tqdm wandb python-dotenv huggingface_hub accelerate safetensors

import pandas as pd

from early_exit_adapters.base_speculative_decode import run_speculative_eval
from early_exit_adapters.data import load_fineweb_stream
from early_exit_adapters.evaluate import layers_kl_over_data_rows
from early_exit_adapters.hf_utils import load_adapters_from_hf, upload_adapter_folder_to_hf
from early_exit_adapters.model import load_lm_model_and_tokenizer
from early_exit_adapters.pipeline import main
from early_exit_adapters.train import train_initial_adapters
from early_exit_adapters.visualization import (
    plot_baseline_all_metrics,
    plot_baseline_metric_by_layer,
    plot_baseline_normalized_metrics,
    plot_baseline_vs_adapted,
)

## 3. Mount Drive and choose where useful artifacts are saved

Checkpoints, final adapters, logs, metadata, and evaluation JSON/JSONL should go somewhere persistent. In Colab, `/content` disappears when the runtime ends; Google Drive survives.

In [ ]:
from pathlib import Path
import os

SAVE_TO_DRIVE = True
RUN_NAME = "qwen35_2b_residual_adapters"

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/early_exit_adapters")
else:
    ARTIFACT_ROOT = Path("/content/artifacts/early_exit_adapters")

OUT_DIR = ARTIFACT_ROOT / RUN_NAME
EVAL_DIR = ARTIFACT_ROOT / f"{RUN_NAME}_eval"
BASELINE_EVAL_DIR = EVAL_DIR / "baseline"
ADAPTED_EVAL_DIR = EVAL_DIR / "adapted"

for path in [OUT_DIR, BASELINE_EVAL_DIR, ADAPTED_EVAL_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Saving training artifacts to:", OUT_DIR)
print("Saving eval artifacts to:", EVAL_DIR)

## 4. Load `.env`

In Colab you can either upload a `.env` file to the repo root, store one in Drive, or set secrets directly in `os.environ` before logging into W&B/Hugging Face.

In [ ]:
from dotenv import load_dotenv

load_dotenv()  # loads .env from the cloned repo root if present

# Optional Colab-only fallback, if you store .env next to the artifacts in Drive:
# load_dotenv(ARTIFACT_ROOT / ".env")

## 5. Load model/tokenizer

In [ ]:
model_name = "Qwen/Qwen3.5-2B"

model, tokenizer, device = load_lm_model_and_tokenizer(model_name=model_name)
device

## 6. Load FineWeb-Edu stream

In [ ]:
dataset = load_fineweb_stream(
    dataset_name="HuggingFaceFW/fineweb-edu",
    split="train",
    seed=42,
    buffer_size=10_000,
)

## 7. Train adapters

In [ ]:
adapters, logs_df = train_initial_adapters(
    model=model,
    tokenizer=tokenizer,
    dataset=dataset,
    candidate_layers=[4, 8, 12, 16, 18, 20],
    max_steps=500,
    seq_len=128,
    lr=1e-4,
    temperature=2.0,
    top_k=5,
    log_every=20,
    eval_step=100,
    eval_size=32,
    save_every=100,
    out_dir=str(OUT_DIR),
    use_wandb=True,
    wandb_project="qwen35-early-exit-adapters",
    wandb_run_name=RUN_NAME,
    wandb_mode=None,
)


## 8. Confirm useful files were saved

In [ ]:
logs_df.tail()

In [ ]:
for path in sorted(OUT_DIR.glob("*")):
    print(path.name, path.stat().st_size, "bytes")

assert (OUT_DIR / "adapters_final.pt").exists(), "Final adapter checkpoint was not saved."
assert (OUT_DIR / "training_logs.jsonl").exists(), "Training logs were not saved."
print("Training artifact check passed:", OUT_DIR)

## 9. Run baseline/adapted layer evaluation

In [ ]:
baseline_dataset = load_fineweb_stream(
    dataset_name="HuggingFaceFW/fineweb-edu",
    split="train",
    seed=42,
    buffer_size=10_000,
)

baseline_records_df, baseline_summary = layers_kl_over_data_rows(
    model=model,
    tokenizer=tokenizer,
    dataset=baseline_dataset,
    candidate_layers=[4, 8, 12, 16, 18, 20, 22, 24],
    seq_len=128,
    num_eval_rows=100,
    temperature=2.0,
    top_k=5,
    out_dir=str(BASELINE_EVAL_DIR),
)

adapted_dataset = load_fineweb_stream(
    dataset_name="HuggingFaceFW/fineweb-edu",
    split="train",
    seed=999,
    buffer_size=10_000,
)

adapted_records_df, adapted_summary = layers_kl_over_data_rows(
    model=model,
    tokenizer=tokenizer,
    dataset=adapted_dataset,
    candidate_layers=[4, 8, 12, 16, 18, 20],
    adapters=adapters,
    run_name="adapted",
    seq_len=128,
    num_eval_rows=100,
    temperature=2.0,
    top_k=5,
    out_dir=str(ADAPTED_EVAL_DIR),
)

compare_summary = pd.concat([baseline_summary, adapted_summary], ignore_index=True)

## 10. Plot baseline metrics

In [ ]:
plot_baseline_metric_by_layer(
    baseline_summary,
    metric="kl_to_teacher",
    title="Baseline KL to final model by layer",
)
plot_baseline_all_metrics(baseline_summary)
plot_baseline_normalized_metrics(baseline_summary)

## 11. Plot baseline vs adapted metrics

In [ ]:
plot_baseline_vs_adapted(compare_summary, metric="kl_to_teacher")
plot_baseline_vs_adapted(compare_summary, metric="ce")
plot_baseline_vs_adapted(compare_summary, metric="top1_teacher_agreement")
plot_baseline_vs_adapted(compare_summary, metric="topk_overlap")
plot_baseline_vs_adapted(compare_summary, metric="accept_proxy_exact")
plot_baseline_vs_adapted(compare_summary, metric="accept_proxy_sampled")

## 12. Run speculative decoding benchmark

This compares baseline early-layer speculative decoding with adapted early-layer speculative decoding. The adapted run uses the adapter from the same layer being evaluated.

In [ ]:
spec_dataset = load_fineweb_stream(
    dataset_name="HuggingFaceFW/fineweb-edu",
    split="train",
    seed=123,
    buffer_size=10_000,
)

baseline_spec_result = run_speculative_eval(
    model=model,
    tokenizer=tokenizer,
    ds=spec_dataset,
    layer_index=8,
    gamma=3,
    seqlen=10,
    num_prefix_tokens=10,
    num_of_examples=100,
    adapter=None,
    draft_temperature=0.8,
    device_type=device,
)

baseline_spec_result

In [ ]:
spec_dataset = load_fineweb_stream(
    dataset_name="HuggingFaceFW/fineweb-edu",
    split="train",
    seed=123,
    buffer_size=10_000,
)

adapted_layer_index = 12

adapted_spec_result = run_speculative_eval(
    model=model,
    tokenizer=tokenizer,
    ds=spec_dataset,
    layer_index=adapted_layer_index,
    gamma=4,
    seqlen=10,
    num_prefix_tokens=10,
    num_of_examples=100,
    adapter=adapters[str(adapted_layer_index)],
    draft_temperature=0.8,
    device_type=device,
)

adapted_spec_result

In [ ]:
spec_results_df = pd.DataFrame([
    {"run_name": "baseline", **baseline_spec_result},
    {"run_name": "adapted", **adapted_spec_result},
])

spec_results_path = EVAL_DIR / "speculative_decode_results.jsonl"
spec_results_df.to_json(spec_results_path, orient="records", lines=True)
print("saved:", spec_results_path)
spec_results_df

## 13. Upload adapters to Hugging Face

In [ ]:
# upload_adapter_folder_to_hf(
#     repo_id="Maorb23/qwen35-2b-early-exit-adapters",
#     folder_path=str(OUT_DIR),
# )

## 14. Reload adapters from Hugging Face for sanity check

In [ ]:
# reloaded_adapters = load_adapters_from_hf(
#     model=model,
#     repo_id="Maorb23/qwen35-2b-early-exit-adapters",
#     filename="adapters_final.pt",
#     device=device,
# )
# reloaded_adapters

## 15. Optional quick smoke test cell

Use this before a long run if you want to verify the whole pipeline path with tiny settings.

In [ ]:
# smoke_adapters, smoke_logs_df = main(
#     model_name="Qwen/Qwen3.5-2B",
#     candidate_layers=[4, 8],
#     max_steps=5,
#     seq_len=128,
#     eval_size=2,
#     eval_step=5,
#     save_every=5,
#     out_dir=str(ARTIFACT_ROOT / "smoke_test"),
#     use_wandb=True,
#     wandb_project="qwen35-early-exit-adapters",
#     wandb_run_name="smoke_test",
#     wandb_mode="offline",
#     upload_to_hf=False,
# )
# smoke_logs_df.tail()